In [ ]:
import sys, os

_notebook_dir = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, os.path.join(_notebook_dir, '..', 'scripts'))

import pandas as pd
from data_utils import PARQUET_RAW as PARQUET, REF

print("PARQUET path:", PARQUET)
print("REF path:", REF)

In [ ]:
sample_info = pd.read_parquet(os.path.join(PARQUET, '9_DepMap_sample_info.parquet'))
print('sample_info shape:', sample_info.shape)
print('columns:', sample_info.columns.tolist())
print(sample_info.head(3))

In [ ]:
cellosaurus = pd.read_parquet(os.path.join(PARQUET, '7_cellosaurus.parquet'))
print('cellosaurus shape:', cellosaurus.shape)
print('columns:', cellosaurus.columns.tolist())
print(cellosaurus.head(3))

In [ ]:
# Step 1: Take only needed columns from sample_info
cell_line_lookup = sample_info[[
    "DepMap_ID",
    "cell_line_name",
    "stripped_cell_line_name",
    "RRID"
]].copy()

# Step 2: Rename to contract spec
cell_line_lookup = cell_line_lookup.rename(columns={
    "DepMap_ID": "model_id",
    "RRID":      "rrid"
})

# Step 3: Slim down cellosaurus to only what we need
cello_slim = cellosaurus[[
    "Accession (CVCL_xxxx)",
    "Synonyms"
]].rename(columns={
    "Accession (CVCL_xxxx)": "cvcl_accession",
    "Synonyms":               "synonyms"
})

# Step 4: Left join — sample_info stays at 1,840 rows, cellosaurus adds columns
cell_line_lookup = cell_line_lookup.merge(
    cello_slim,
    left_on="rrid",
    right_on="cvcl_accession",
    how="left"
)

# Step 5: Convert semicolon-delimited synonyms → pipe-delimited
cell_line_lookup["synonyms"] = (
    cell_line_lookup["synonyms"]
    .fillna("")
    .str.strip()
    .str.replace(r";\s*", "|", regex=True)
    .replace("", float("nan"))
)

# Step 6: Force plain object dtype
cell_line_lookup = cell_line_lookup.astype(object)

# Step 7: Final column order per contract
cell_line_lookup = cell_line_lookup[[
    "model_id", "cell_line_name", "stripped_cell_line_name",
    "cvcl_accession", "synonyms", "rrid"
]]

print(cell_line_lookup.shape)
print(cell_line_lookup.head(5))

In [ ]:
print("=== VALIDATION ===")
print(f"Total rows:                    {len(cell_line_lookup):,}")
print(f"Null model_id:                 {cell_line_lookup['model_id'].isna().sum()}")
print(f"Duplicate model_id:            {cell_line_lookup['model_id'].duplicated().sum()}")
print(f"Null cell_line_name:           {cell_line_lookup['cell_line_name'].isna().sum()}")
print(f"Has cvcl_accession:            {cell_line_lookup['cvcl_accession'].notna().sum():,}")
print(f"Has synonyms:                  {cell_line_lookup['synonyms'].notna().sum():,}")
print(f"No cellosaurus match:          {cell_line_lookup['cvcl_accession'].isna().sum():,}")

In [ ]:
OUT = os.path.join(REF, "cell_line_lookup.parquet")

cell_line_lookup.to_parquet(OUT, index=False, engine="fastparquet")

confirm = pd.read_parquet(OUT, engine="fastparquet")
print(f"Saved and verified: {confirm.shape}")
print(f"File size: {os.path.getsize(OUT):,} bytes")